### **PT.1 - Código Original Professor:**

In [292]:
docs <- c(
    d1 = "recuperacao de informacao ordena documentos por relevancia",
    d2 = "o modelo de espaco vetorial representa documentos como vetores",
    d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
    d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
    d5 = "o indice invertido acelera a busca em muitos documentos",
    d6 = "embeddings capturam a semantica de palavras e documentos",
    d7 = "a avaliacao mede a relevancia dos resultados da busca",
    d8 = "ciencia de dados combina estatistica e programacao"
)

tok <- function(x) unlist(strsplit(tolower(x), "\\s+"))
tokens <- lapply(docs, tok)
vocab <- sort(unique(unlist(tokens)))
tdm <- sapply(tokens, function(t) as.integer(table(factor(t, levels = vocab))))
rownames(tdm) <- vocab
dim(tdm)


[1] 45  8

In [293]:
tf <- tdm
N <- ncol(tdm)
df <- rowSums(tdm > 0)
idf <- log(N / df)
w <- tf * idf # matriz TF-IDF (termos x documentos)
round(w[c("documentos", "modelo", "de"), ], 2)

,d1,d2,d3,d4,d5,d6,d7,d8
documentos,0.69,0.69,0.00,0,0.69,0.69,0,0.00
modelo,0.00,1.39,1.39,0,0.00,0.00,0,0.00
de,0.47,0.47,0.94,0,0.00,0.47,0,0.47


In [294]:
norm_cols <- function(m) sweep(m, 2, sqrt(colSums(m^2)), "/")
wn <- norm_cols(w)
round(colSums(wn^2), 2) # comprimento^2 de cada documento

d1 d2 d3 d4 d5 d6 d7 d8 
 1  1  1  1  1  1  1  1

In [295]:
cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))
consulta <- "modelo de recuperacao"
q <- as.integer(table(factor(tok(consulta), levels = vocab)))
qw <- q * idf
round(qw[qw > 0], 2)

de      modelo recuperacao 
       0.47        1.39        1.39

In [296]:
scores <- apply(w, 2, function(dvec) cosseno(qw, dvec))
round(sort(scores, decreasing = TRUE), 3)

d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000

In [297]:
melhor <- names(which.max(scores))
docs[melhor]

d1 
"recuperacao de informacao ordena documentos por relevancia"

### **PT.2 - Código modificado usando Modularização SPR (Single Responsability Principle):**

Fiz isso para facilitar meu entendimento do código original. Adicionei a funcionalidade de **remoção de stopwords**. Ela pode ser desativada. Ela é a única funcionalidade que utiliza libraries, o restante usa apenas R nativo.

In [298]:
if(!require(pacman)) install.packages(pacman)
pacman::p_load(stopwords)

In [299]:
docs <- c(
    d1 = "recuperacao de informacao ordena documentos por relevancia",
    d2 = "o modelo de espaco vetorial representa documentos, como vetores!!!",
    # d3 = "I love my brother! Did you hear me?"
    d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
    d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
    d5 = "o indice invertido acelera a busca em muitos documentos",
    d6 = "embeddings capturam a semantica de palavras e documentos",
    d7 = "a avaliacao mede a relevancia dos resultados da busca",
    d8 = "ciencia de dados combina estatistica e programacao"
)

In [300]:
# ====================================================================
# 1. MÓDULO DE PRÉ-PROCESSAMENTO (TEXTO -> TOKENS)
# ====================================================================

remover__pontuacao__ <- function(txt, remover_stopwords = FALSE) {
  txt <- gsub("[[:punct:]]", "", txt)
  return(txt)
}

tokenizar_texto <- function(texto) {
  unlist(strsplit(tolower(
    remover__pontuacao__(texto)), "\\s+"))
}

remover__stopwords__ <- function(txt) {
  token_from_txt <- tokenizar_texto(txt)
  
  stopwords_set_1 <- stopwords::stopwords("en", source = "snowball")
  stopwords_set_2 <- stopwords::stopwords("pt", source = "snowball")
  stopwords_set_duplo <- unique(c(stopwords_set_1, stopwords_set_2))
 
  return (token_from_txt[!token_from_txt %in% stopwords_set_duplo])
}

tokenizar_corpus <- function(corpus, remover_stopwords = FALSE) {
  if (remover_stopwords == FALSE)
    return (lapply(corpus, tokenizar_texto))
  return (lapply(corpus, remover__stopwords__))
}

criar_vocabulario <- function(tokens_corpus) {
  sort(unique(unlist(tokens_corpus)))
}

In [301]:
# ====================================================================
# 2. MÓDULO MATEMÁTICO (VETORES E MATRIZES)
# ====================================================================

gerar_pesos_tfidf <- function(tokens_corpus, vocabulario) {
  
  matriz_tf <- sapply(tokens_corpus, function(t) {       # 1. Matriz TF (Frequência dos Termos)
    as.integer(table(factor(t, levels = vocabulario)))
  })
  rownames(matriz_tf) <- vocabulario
  
  N <- ncol(matriz_tf)                                   # 2. Vetor IDF (Frequência Inversa)
  df <- rowSums(matriz_tf > 0)
  vetor_idf <- log(N / df)
  
  matriz_tfidf <- matriz_tf * vetor_idf                  # 3. Matriz TF-IDF Final
  
  return(list(matriz = matriz_tfidf, idf = vetor_idf))   # Retorna a matriz pronta e o
                                                         # vetor idf (vamos precisar dele para a consulta)
}

# Cálculo do cosseno mantido de forma simples, mas com proteção contra divisão por zero
calcular_cosseno <- function(a, b) {
  if (sum(a) == 0 || sum(b) == 0) return(0)
  sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))
}

In [302]:
# ====================================================================
# 3. MÓDULO PRINCIPAL (A BUSCA)
# ====================================================================

realizar_busca <- function(corpus, consulta, remover_stopwords = FALSE) {
  
  # Passo 1: Usa as suas novas funções para preparar os dados
  tokens_corpus <- tokenizar_corpus(corpus, remover_stopwords)
  vocabulario <- criar_vocabulario(tokens_corpus)
  
  # Passo 2: Gera a matriz de pesos dos documentos
  pesos <- gerar_pesos_tfidf(tokens_corpus, vocabulario)
  
  # Passo 3: Prepara a consulta garantindo que ela use a SUA regra de stopwords
  if (remover_stopwords == TRUE) {
    tokens_consulta <- remover__stopwords__(consulta)
  } else {
    tokens_consulta <- tokenizar_texto(consulta)
  }
  
  # Passo 4: Transforma a consulta em vetor usando o vocabulário e o IDF do corpus
  tf_consulta <- as.integer(table(factor(tokens_consulta, levels = vocabulario)))
  vetor_tfidf_consulta <- tf_consulta * pesos$idf
  
  # Passo 5: Calcula o cosseno entre a consulta e todos os documentos da matriz
  scores <- apply(pesos$matriz, 2, function(vetor_doc) {
    calcular_cosseno(vetor_tfidf_consulta, vetor_doc)
  })
  
  # Passo 6: Retorna os resultados organizados
  indice_melhor <- which.max(scores)
  
  list(
    configuracao = paste("Stopwords removidas:", remover_stopwords),
    melhor_documento = corpus[indice_melhor],
    score = round(scores[indice_melhor], 3),
    ranking = round(sort(scores, decreasing = TRUE), 3)
  )
}

### **PT.3 - Busca com base no código modificado:**
Teste com os dois comportamentos principais possívels, com e sem remoção de stopwords.

In [303]:
# Consulta com stopwords mantidas (Comportamento Original)

busca_normal <- realizar_busca(
    corpus = docs, 
    consulta = "o modelo de recuperacao da informacao", 
    remover_stopwords = FALSE)

print(busca_normal$melhor_documento) # Resultados idênticos (É o esperado) ...

                                                          d1 
"recuperacao de informacao ordena documentos por relevancia" 


In [304]:
# Consulta com REMOCAO DE STOPWORDS 
# (Ignora "o", "de", "da")

busca_otimizada <- realizar_busca(
    corpus = docs, 
    consulta = "o modelo de recuperacao da informacao", 
    remover_stopwords = TRUE)

print(busca_otimizada$melhor_documento) # Resultados idênticos (É o esperado) ...

                                                          d1 
"recuperacao de informacao ordena documentos por relevancia" 


### **PT.4 - Web Scrapping:**